# Notebook 32 — MJO 3-D Disentangled Latent (L2 phase circle + ENSO axis)
**Project:** ENSO-BSISO SSL — MJO moisture-constraint experiment
**Author:** Jiayi (jh9141@nyu.edu)

nb31 showed the 2-D wall: with no L2 norm the angle isn't a clean phase (circ_corr ~0.2), but L2-norm in 2-D
would kill the radial ENSO axis. **3-D resolves it.** The encoder outputs `z = (z1, z2, z3)`:
- **phase circle** = `(z1, z2)` **L2-normalized** -> on the unit circle, so `theta = atan2` is a *true* angular
  phase by construction. Temporal-contrastive (cosine InfoNCE on day t vs t+/-<=3) organizes the cycle.
- **ENSO axis** = `z3`, free. Because normalization discards the magnitude of (z1,z2), the slow
  amplitude/ENSO envelope is *forced* onto z3.

`L = InfoNCE_cosine(circle_t, circle_{t+/-3})  +  lam_aux*(MSE q_col + MSE OLR from z)  +  lam_var*var_floor(z3)`

Then measure delta_theta(q_col, conv) in the **circle angle** (should finally match own-RMM's moisture-mode
lead) and check ENSO separation along z3.

Inputs as nb31. Outputs: `MJO/moisture_constraints/results/aux3d/`.

---

## Cell 1 — Setup + config

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os, json, time
import numpy as np, pandas as pd
import torch, torch.nn as nn, torch.nn.functional as F, torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import matplotlib.pyplot as plt

PROJECT_DIR='/content/drive/MyDrive/BSISO_SSL_Project'; MJO_DIR=f'{PROJECT_DIR}/MJO'
PROC=f'{MJO_DIR}/data/processed'; MOIST=f'{MJO_DIR}/moisture_constraints/data/processed'
OUT=f'{MJO_DIR}/moisture_constraints/results/aux3d'; os.makedirs(OUT, exist_ok=True)

LATENT_DIM   = 3             # (z1,z2)=phase circle, z3=ENSO axis
TAU_CIRC     = 0.2           # cosine InfoNCE temperature on the unit circle
EPOCHS       = 80
BATCH_SIZE   = 256
LR           = 1e-3
WEIGHT_DECAY = 1e-4
MAX_DELTA    = 3
LAM_AUX      = 0.5           # moisture/OLR auxiliary (shapes z3 + ties to physics)
LAM_VAR      = 1.0           # variance floor on z3 (keep ENSO axis alive)
SEED=42; VAL_STRIDE=5
torch.manual_seed(SEED); np.random.seed(SEED)
device=torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device', device, ' tau_circ', TAU_CIRC, ' lam_aux', LAM_AUX, ' lam_var', LAM_VAR)

## Cell 2 — Load SSL input + aligned q_col/OLR targets + own-RMM + year split (same as nb31)

In [ ]:
X_bp=np.load(f'{PROC}/X_MJO_bp20_90.npy'); labels=pd.read_csv(f'{PROC}/labels_aligned_mjo_bp20_90.csv',parse_dates=['date'])
lons=np.load(f'{PROC}/longitudes_mjo.npy'); M=len(X_bp); assert len(labels)==M
bpdates=pd.DatetimeIndex(labels['date']).normalize()
qcol_full=np.load(f'{MOIST}/qcol_mjo_processed.npy')
full=pd.read_csv(f'{PROC}/labels_aligned_mjo.csv',parse_dates=['date'])
full_row={d:i for i,d in enumerate(pd.DatetimeIndex(full['date']).normalize())}
fi=np.array([full_row.get(d,-1) for d in bpdates])
qcol_bp=np.full((M,len(lons)),np.nan,np.float32); ok=fi>=0; qcol_bp[ok]=qcol_full[fi[ok]]
olr_bp=X_bp[:,1,0,:].astype(np.float32)
q_finite=np.isfinite(qcol_bp).all(1); qcol_bp=np.nan_to_num(qcol_bp).astype(np.float32)
pcs_full=np.load(f'{PROC}/mjo_rmm_own_pcs.npy')
pcs_bp=np.where(fi[:,None]>=0, pcs_full[np.clip(fi,0,None)], np.nan)
theta_own_bp=np.arctan2(pcs_bp[:,1],pcs_bp[:,0])
years=bpdates.year.values; val_years=sorted(np.unique(years))[::VAL_STRIDE]
train_idx=np.where(~np.isin(years,val_years))[0]
print(f'M={M}  q covered {int(q_finite.sum())}  train {len(train_idx)}')

## Cell 3 — Encoder (3D) + decoders + pair sampler + losses

In [ ]:
class MJOEncoder(nn.Module):
    def __init__(s, d=3):
        super().__init__()
        s.c1=nn.Conv2d(3,16,(1,3),padding=(0,1),bias=False); s.b1=nn.BatchNorm2d(16); s.p1=nn.MaxPool2d((1,2))
        s.c2=nn.Conv2d(16,32,(1,3),padding=(0,1),bias=False); s.b2=nn.BatchNorm2d(32); s.p2=nn.MaxPool2d((1,2))
        s.c3=nn.Conv2d(32,32,(1,3),padding=(0,1),bias=False); s.b3=nn.BatchNorm2d(32)
        s.gp=nn.AdaptiveAvgPool2d(1); s.fc=nn.Linear(32,d)
    def forward(s,x):
        x=s.p1(F.relu(s.b1(s.c1(x)))); x=s.p2(F.relu(s.b2(s.c2(x)))); x=F.relu(s.b3(s.c3(x)))
        return s.fc(s.gp(x).view(x.size(0),-1))

class FieldDecoder(nn.Module):
    def __init__(s,d=3,nlon=180): super().__init__(); s.net=nn.Sequential(nn.Linear(d,64),nn.ReLU(True),nn.Linear(64,nlon))
    def forward(s,z): return s.net(z)

class TemporalPairSampler:
    def __init__(s, labels_df, allowed, md=3):
        s.labels=labels_df; s.md=md; sub=labels_df.loc[list(allowed)]
        s.d2i={pd.Timestamp(d):int(i) for d,i in zip(sub['date'].values,sub.index.values)}
    def positive(s,a):
        ad=s.labels.loc[a,'date']; c=[]
        for dl in range(-s.md,s.md+1):
            if dl==0: continue
            t=ad+pd.Timedelta(days=dl)
            if t.year==ad.year and pd.Timestamp(t) in s.d2i: c.append(s.d2i[pd.Timestamp(t)])
        return (a,int(np.random.choice(c))) if c else (a,a)

class AuxDataset(Dataset):
    def __init__(s,X,q,qf,labels_df,idx,md=3): s.X=X;s.q=q;s.qf=qf;s.idx=np.asarray(idx);s.s=TemporalPairSampler(labels_df,idx,md)
    def __len__(s): return len(s.idx)
    def __getitem__(s,i):
        a=int(s.idx[i]); a,b=s.s.positive(a)
        return (torch.from_numpy(s.X[a]).float(),torch.from_numpy(s.X[b]).float(),
                torch.from_numpy(s.q[a]).float(),torch.tensor(float(s.qf[a])))

def infonce_cos(zA,zB,tau):                       # zA,zB are L2-normalized -> cosine InfoNCE
    sim=(zA@zB.T)/tau
    return F.cross_entropy(sim, torch.arange(zA.size(0),device=zA.device))
def var_floor(z, gamma=1.0, eps=1e-4):
    return torch.mean(F.relu(gamma-torch.sqrt(z.var(0)+eps)))

loader=DataLoader(AuxDataset(X_bp,qcol_bp,q_finite,labels,train_idx,MAX_DELTA),
                  batch_size=BATCH_SIZE,shuffle=True,num_workers=2,pin_memory=(device.type=='cuda'),drop_last=True)
print('batches/epoch', len(loader))

## Cell 4 — Train (cosine InfoNCE on the circle + moisture/OLR aux + z3 variance floor)

In [ ]:
enc=MJOEncoder(LATENT_DIM).to(device)
qdec=FieldDecoder(LATENT_DIM,len(lons)).to(device); odec=FieldDecoder(LATENT_DIM,len(lons)).to(device)
opt=optim.Adam(list(enc.parameters())+list(qdec.parameters())+list(odec.parameters()),lr=LR,weight_decay=WEIGHT_DECAY)
sch=optim.lr_scheduler.CosineAnnealingLR(opt,T_max=EPOCHS,eta_min=1e-5)
hist={'loss':[],'nce':[],'aux':[],'var':[]}; t0=time.time()
for ep in range(EPOCHS):
    enc.train(); qdec.train(); odec.train(); agg={'loss':0,'nce':0,'aux':0,'var':0}; nb=0
    for xa,xb,qa,qf in loader:
        xa,xb,qa,qf=xa.to(device),xb.to(device),qa.to(device),qf.to(device)
        za=enc(xa); zb=enc(xb)
        zca=F.normalize(za[:,:2],dim=1); zcb=F.normalize(zb[:,:2],dim=1)   # phase circle
        L_nce=infonce_cos(zca,zcb,TAU_CIRC)
        L_var=var_floor(za[:,2:3])                                        # keep ENSO axis alive
        qhat=qdec(za); ohat=odec(za); otgt=xa[:,1,0,:]
        w=qf.clamp(min=0); denom=w.sum().clamp(min=1)
        L_aux=((qhat-qa).pow(2).mean(1)*w).sum()/denom + F.mse_loss(ohat,otgt)
        L=L_nce+LAM_AUX*L_aux+LAM_VAR*L_var
        opt.zero_grad(); L.backward(); opt.step()
        agg['loss']+=L.item();agg['nce']+=L_nce.item();agg['aux']+=L_aux.item();agg['var']+=L_var.item();nb+=1
    sch.step()
    for k in agg: hist[k].append(agg[k]/nb)
    if (ep+1)%10==0 or ep<3:
        print(f'ep {ep+1:3d}/{EPOCHS} loss={hist["loss"][-1]:.3f} nce={hist["nce"][-1]:.3f} '
              f'aux={hist["aux"][-1]:.3f} var={hist["var"][-1]:.3f} lr={sch.get_last_lr()[0]:.1e}')
torch.save(enc.state_dict(),f'{OUT}/encoder_aux3d.pth'); json.dump(hist,open(f'{OUT}/training_history.json','w'))
print(f'Done {(time.time()-t0)/60:.1f} min  (nce should DECREASE toward ln(256)~5.55 then below)')

## Cell 5 — Extract 3D latent; phase = circle angle, ENSO axis = z3

In [ ]:
enc.eval(); Z=np.zeros((M,LATENT_DIM),np.float32)
with torch.no_grad():
    for s in range(0,M,256): Z[s:s+256]=enc(torch.from_numpy(X_bp[s:s+256]).float().to(device)).cpu().numpy()
np.save(f'{OUT}/embeddings.npy', Z)
zc=Z[:,:2]/ (np.linalg.norm(Z[:,:2],axis=1,keepdims=True)+1e-9)
theta_z=np.arctan2(zc[:,1],zc[:,0]); z3=Z[:,2]
print('z3 (ENSO axis) std', round(float(z3.std()),3), ' circle radius range',
      round(float(np.linalg.norm(Z[:,:2],axis=1).min()),2),'..',round(float(np.linalg.norm(Z[:,:2],axis=1).max()),2))

## Cell 6 — Diagnostics: delta_theta in the phase circle + ENSO on z3

In [ ]:
REGIONS={'IndianOcean':(60,90),'MaritimeContinent':(100,130),'WestPacific':(140,170)}
rmask={k:(lons>=v[0])&(lons<=v[1]) for k,v in REGIONS.items()}
phase=labels['phase'].values.astype(int); amp=labels['amplitude'].values
enso=labels['enso_category'].values; weak=labels['weak_mjo'].values.astype(bool)
active=(~weak)&(amp>=1.0)&q_finite; conv=-olr_bp; ph_ang=(phase-1)/8.0*2*np.pi
def circ_corr(a,b):
    a=a-np.angle(np.mean(np.exp(1j*a))); b=b-np.angle(np.mean(np.exp(1j*b)))
    return float(np.sum(np.sin(a)*np.sin(b))/np.sqrt(np.sum(np.sin(a)**2)*np.sum(np.sin(b)**2)+1e-12))
def wrapdeg(a): return np.degrees((a+np.pi)%(2*np.pi)-np.pi)
def offsets(theta,mask):
    if circ_corr(theta[mask],ph_ang[mask])<0: theta=-theta
    e=np.exp(1j*theta[mask]); out={}
    for rk,rm in rmask.items():
        Af=np.nanmean(qcol_bp[mask]*e[:,None],0)[rm].sum(); Ac=np.nanmean(conv[mask]*e[:,None],0)[rm].sum()
        out[rk]=float(round(wrapdeg(np.angle(Af)-np.angle(Ac)),1))
    return out, float(round(circ_corr(theta[mask],ph_ang[mask]),2))
oz,cz=offsets(theta_z,active); oo,co=offsets(theta_own_bp,active)
print('delta_theta(q_col,conv):')
print(f'  aux3d circle:  {oz}   circ_corr={cz}   (target: match own-RMM)')
print(f'  own-RMM:       {oo}   circ_corr={co}')

# ENSO: does z3 separate El Nino / La Nina?  + full-3D displacement z
import numpy as _np
zr=z3[active]; en=enso[active]
mEN=en=='El Nino'; mLN=en=='La Nina'
print(f'\nz3 (ENSO axis) mean: El Nino {zr[mEN].mean():+.2f}  La Nina {zr[mLN].mean():+.2f}  '
      f'separation {abs(zr[mEN].mean()-zr[mLN].mean()):.2f} (in std units {abs(zr[mEN].mean()-zr[mLN].mean())/zr.std():.2f})')
def enso_z(emb):
    idx=np.where(active)[0]; e=emb[idx]; lab=labels.iloc[idx].reset_index(drop=True); obs=[]
    for p in range(1,9):
        a=(lab['phase']==p)&(lab['enso_category']=='El Nino'); b=(lab['phase']==p)&(lab['enso_category']=='La Nina')
        if a.sum()<3 or b.sum()<3: continue
        obs.append(np.linalg.norm(e[a.values].mean(0)-e[b.values].mean(0)))
    rng=np.random.default_rng(0); base=[]
    for _ in range(200):
        sh=lab['enso_category'].sample(frac=1,random_state=rng.integers(1e6)).values; t=[]
        for p in range(1,9):
            mph=(lab['phase']==p).values; a=mph&(sh=='El Nino'); b=mph&(sh=='La Nina')
            if a.sum()<3 or b.sum()<3: continue
            t.append(np.linalg.norm(e[a].mean(0)-e[b].mean(0)))
        if t: base.append(np.mean(t))
    return float((np.mean(obs)-np.mean(base))/(np.std(base)+1e-8))
print(f'ENSO displacement z (full 3D): {enso_z(Z):.2f}')

sub=np.random.default_rng(0).choice(np.where(active)[0],min(5000,int(active.sum())),replace=False)
hsv=plt.cm.hsv(np.linspace(0,1,9)); epal={'El Nino':'#d62728','Neutral':'#7f7f7f','La Nina':'#1f77b4'}
fig,ax=plt.subplots(1,3,figsize=(18,5.2))
for p in range(1,9):
    m=sub[phase[sub]==p]; ax[0].scatter(zc[m,0],zc[m,1],s=6,alpha=.5,color=hsv[p-1],label=f'P{p}')
ax[0].set_title('phase circle (z1,z2 normalized) by RMM phase'); ax[0].legend(fontsize=6,ncol=2); ax[0].set_aspect('equal')
for c in epal:
    m=sub[enso[sub]==c]; ax[1].scatter(theta_z[m]*180/np.pi, z3[m], s=6, alpha=.5, color=epal[c], label=c)
ax[1].set_xlabel('phase angle (deg)'); ax[1].set_ylabel('z3 (ENSO axis)'); ax[1].set_title('z3 vs phase, by ENSO'); ax[1].legend(fontsize=7)
ax[2].hist(z3[active][mEN],bins=40,alpha=.5,color='#d62728',label='El Nino',density=True)
ax[2].hist(z3[active][mLN],bins=40,alpha=.5,color='#1f77b4',label='La Nina',density=True)
ax[2].set_xlabel('z3 (ENSO axis)'); ax[2].set_title('z3 distribution by ENSO'); ax[2].legend(fontsize=8)
fig.suptitle(f'aux3d: phase circle (circ_corr {cz}) + ENSO axis (z {enso_z(Z):.1f})',fontweight='bold')
plt.tight_layout(); p=f'{OUT}/aux3d_latent.png'; plt.savefig(p,dpi=130,bbox_inches='tight'); plt.show(); print('Saved',p)
json.dump({'circ_corr_aux3d':cz,'delta_theta_aux3d':oz,'delta_theta_own_rmm':oo,'enso_z':round(enso_z(Z),2)},
          open(f'{OUT}/aux3d_summary.json','w'),indent=2,default=float)
print('Saved', f'{OUT}/aux3d_summary.json')

---
## Done!
Outputs in `MJO/moisture_constraints/results/aux3d/`. **Win condition:** the circle's `circ_corr(theta,phase)`
jumps toward own-RMM's (~0.6-0.9) because the angle is now a *true* phase, the circle Δθ(q_col) matches
own-RMM's moisture-mode lead, AND El Nino/La Nina separate along z3. That's the disentangled, physically
readable MJO latent: angle = moisture/convection cycle, z3 = ENSO modulation.

---
*DDCS Project | jh9141@nyu.edu*